# SuperAI Engineer OCR 2569 (EasyOCR Refactored Pipeline)
Offline Colab pipeline using EasyOCR and OpenCV to extract votes from election documents.


**Pipeline Steps:**
1. Group image files by Document ID
2. Image Preprocessing (Crop, CLAHE, Erosion)
3. Extract text and convert to Data Objects (`TextBox`)
4. Match parties and assign votes using heuristics (X, Y coordinates and Fuzzy Matching)
5. Generate the final CSV file for Kaggle submission

## Cell 1: Install Dependencies

In [1]:
!pip install easyocr pythainlp python-Levenshtein pandas tqdm rapidfuzz natsort opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 31.9 MB/s eta 0:00:00


## Cell 2: Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 3: Imports & Global Configuration

In [3]:
from __future__ import annotations

import os
import re
import cv2
import numpy as np
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from pythainlp.util import thaiword_to_num
from natsort import natsorted
from rapidfuzz import fuzz
import easyocr

# ==========================================
# Paths Configuration
# ==========================================
BASE_DIR = Path('/content/drive/MyDrive/final_data')
IMAGE_DIR = BASE_DIR / "images"
SUBMISSION_PATH = BASE_DIR / "submission_template.csv"
SUBMISSION_OUTPUT = BASE_DIR / "submission_easyocr_refactored.csv"

# ==========================================
# Initialize OCR Model
# ==========================================
print("กำลังโหลดโมเดล EasyOCR (GPU)...")
reader = easyocr.Reader(['th', 'en'], gpu=True)
print("โหลดโมเดลสำเร็จ!")

กำลังโหลดโมเดล EasyOCR (GPU)...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Completeโหลดโมเดลสำเร็จ!


## Cell 4: Data Structures (Dataclasses)
กำหนดโครงสร้างข้อมูลที่ชัดเจน แทนการใช้ Dictionary แบบเดิม เพื่อป้องกันความสับสนของตัวแปร

In [4]:
@dataclass(frozen=True)
class PageInfo:
    doc_id: str
    page_num: int
    path: Path

@dataclass
class TextBox:
    raw_text: str
    clean_text: str
    x_center: float
    y_center: float
    box_height: float
    is_number: bool
    num_value: int | None
    party_match: str | None = None

## Cell 5: Text & Image Utility Functions
แยกฟังก์ชันทำความสะอาดข้อความ และฟังก์ชันประมวลผลภาพ (Image Preprocessing) ออกจากกัน

In [5]:
def remove_table_lines(image: np.ndarray) -> np.ndarray:
    """ลบเส้นตารางแนวตั้งและแนวนอนด้วย OpenCV Morphological Operations"""
    # ตรวจสอบว่าเป็นภาพสีหรือขาวดำ
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image.copy()

    # 1. แปลงเป็น Binary (พื้นดำ ตัวหนังสือ/เส้นเป็นสีขาว) เพื่อสกัดเส้น
    _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)

    # 2. กำหนดขนาด Kernel (ถ้าเส้นขาด ให้เพิ่มเลข 40 ให้มากขึ้น)
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))

    # 3. สกัดเส้นออกมา
    horizontal_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    vertical_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel, iterations=2)

    # 4. รวมเส้นแนวนอนและแนวตั้ง
    table_lines = cv2.add(horizontal_lines, vertical_lines)

    # 5. ขยายขนาดเส้นที่จับได้เล็กน้อยเพื่อให้เวลาลบทิ้งมันเนียนขึ้น (Dilation)
    kernel = np.ones((3,3), np.uint8)
    table_lines = cv2.dilate(table_lines, kernel, iterations=1)

    # 6. ถมสีขาว (255) ทับลงไปตรงจุดที่เป็นเส้นตารางบนภาพต้นฉบับ
    clean_image = gray.copy()
    clean_image[table_lines == 255] = 255

    return clean_image


def extract_numeric_value(text: str) -> int | None:
    clean_text = str(text).replace(' ', '').replace('\u200b', '')
    thai_map = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
    num_text = clean_text.translate(thai_map).replace(',', '')
    replacements = {'O': '0', 'o': '0', 'l': '1', 'I': '1', 'i': '1', 'S': '5', 's': '5', 'B': '8', 'q': '9'}
    for old, new in replacements.items():
        num_text = num_text.replace(old, new)

    digits = re.sub(r'[^\d]', '', num_text)
    if digits:
        return int(digits)

    only_thai = re.sub(r'[^ก-๙]', '', text)
    if only_thai:
        try:
            return int(thaiword_to_num(only_thai))
        except ValueError:
            pass
    return None


def preprocess_document_image(image_path: Path) -> np.ndarray:
    """Preprocess ขั้นสูง: ครอป -> ลบเส้นตาราง -> CLAHE -> Erosion"""
    img = cv2.imread(str(image_path))
    if img is None:
        return np.zeros((100, 100, 3), dtype=np.uint8)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        max_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(max_contour)
        padding = 10
        x_start, y_start = max(0, x - padding), max(0, y - padding)
        x_end, y_end = min(img.shape[1], x + w + padding), min(img.shape[0], y + h + padding)
        img = img[y_start:y_end, x_start:x_end]

    no_lines_img = remove_table_lines(img)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced_img = clahe.apply(no_lines_img)

    kernel = np.ones((2,2), np.uint8)
    thick_img = cv2.erode(enhanced_img, kernel, iterations=1)

    return thick_img

## Cell 6: OCR & Matching Logic
รับภาพที่ผ่านการ Preprocess แล้ว ให้ EasyOCR สกัดเป็น Dataclass `TextBox` และจับคู่ตามพิกัด 2 มิติ

In [6]:
def extract_boxes_from_image(image: np.ndarray, expected_parties: list[str]) -> list[TextBox]:
    results = reader.readtext(image, width_ths=0.7, paragraph=False, mag_ratio=1.5)
    boxes: list[TextBox] = []

    for (bbox, text, prob) in results:
        y_center = (bbox[0][1] + bbox[2][1]) / 2
        x_center = (bbox[0][0] + bbox[1][0]) / 2


        box_height = bbox[2][1] - bbox[0][1]

        clean_text = str(text).replace(' ', '').replace('\u200b', '')
        num_val = extract_numeric_value(text)
        is_number = num_val is not None

        party_match = None
        if not is_number:
            highest_score = 0
            for p in expected_parties:
                score = fuzz.partial_ratio(p, clean_text)
                if score > highest_score:
                    highest_score = score
                    party_match = p
            if highest_score < 85:
                party_match = None

        boxes.append(TextBox(
            raw_text=text,
            clean_text=clean_text,
            x_center=x_center,
            y_center=y_center,
            box_height=box_height,
            is_number=is_number,
            num_value=num_val,
            party_match=party_match
        ))

    return boxes


def map_parties_to_votes(pages: list[PageInfo], expected_parties: list[str]) -> dict[str, int]:
    all_entries: dict[str, int] = {}

    for page in pages:
        processed_img = preprocess_document_image(page.path)
        boxes = extract_boxes_from_image(processed_img, expected_parties)

        party_boxes = [b for b in boxes if b.party_match is not None]
        number_boxes = [b for b in boxes if b.is_number]

        for pb in party_boxes:
            y_tolerance = pb.box_height * 0.8

            candidates = [
                nb for nb in number_boxes
                if abs(nb.y_center - pb.y_center) < y_tolerance and nb.x_center > pb.x_center
            ]

            if candidates:
                candidates.sort(key=lambda b: (abs(b.y_center - pb.y_center), -b.x_center))
                all_entries[pb.party_match] = candidates[0].num_value

    return all_entries

## Cell 7: Document Grouping & Main Pipeline
ฟังก์ชันจัดกลุ่มหน้าเอกสารเข้าด้วยกัน และฟังก์ชันรันลูปหลักเพื่อสร้าง CSV

In [7]:
def group_document_pages(images_dir: Path) -> dict[str, list[PageInfo]]:
    """จัดกลุ่มไฟล์ภาพตาม doc_id อย่างรวดเร็ว (O(N))"""
    grouped: dict[str, list[PageInfo]] = defaultdict(list)
    sorted_paths = natsorted(images_dir.glob("*.png"), key=lambda x: x.stem)

    for path in sorted_paths:
        stem = path.stem
        if "_page" in stem:
            doc_id, page_str = stem.split("_page")
            grouped[doc_id].append(PageInfo(doc_id=doc_id, page_num=int(page_str), path=path))
        else:
            grouped[stem].append(PageInfo(doc_id=stem, page_num=1, path=path))

    for pages in grouped.values():
        pages.sort(key=lambda item: item.page_num)

    return grouped

def build_submission_pipeline():
    print(f"📥 โหลด Template จาก: {SUBMISSION_PATH}")
    df_sub = pd.read_csv(SUBMISSION_PATH)

    party_col = [c for c in df_sub.columns if 'party' in c.lower()][0]
    grouped_pages = group_document_pages(IMAGE_DIR)

    submission_rows: list[dict[str, str]] = []
    doc_ids = df_sub['doc_id'].unique()

    for doc_id in tqdm(doc_ids, desc="🚀 สแกนและดึงคะแนน"):
        df_doc = df_sub[df_sub['doc_id'] == doc_id]
        doc_pages = grouped_pages.get(doc_id, [])

        if not doc_pages:
            for _, row in df_doc.iterrows():
                submission_rows.append({'id': row['id'], 'votes': "0"})
            continue

        # คลีนชื่อพรรคเพื่อส่งให้ OCR ควานหาเพียงรอบเดียว
        expected_parties = df_doc[party_col].astype(str).str.replace(' ', '').tolist()

        # 🎯 Core Extraction
        ocr_results = map_parties_to_votes(doc_pages, expected_parties)

        # หยอดคะแนนลงแถว
        for i, row in df_doc.reset_index(drop=True).iterrows():
            clean_party_name = expected_parties[i]
            vote = str(ocr_results.get(clean_party_name, 0))
            submission_rows.append({'id': row['id'], 'votes': vote})

    # บันทึกไฟล์ตามรูปแบบ Kaggle
    final_sub = pd.DataFrame(submission_rows)[['id', 'votes']]
    SUBMISSION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    final_sub.to_csv(SUBMISSION_OUTPUT, index=False)

    filled = (final_sub['votes'] != '0').sum()
    total = len(final_sub)
    print(f"\n✅ เสร็จสมบูรณ์! ไฟล์ถูกเซฟไว้ที่: {SUBMISSION_OUTPUT}")
    print(f"📊 สรุป: ดึงคะแนนสำเร็จ {filled} แถว (คิดเป็น {filled/total*100:.2f}%) จากทั้งหมด {total} แถว")

## Cell 8: Execute Pipeline
รัน Cell นี้เพื่อเริ่มต้นการทำงานทั้งหมด

In [8]:
build_submission_pipeline()

📥 โหลด Template จาก: /content/drive/MyDrive/final_data/submission_template.csv


🚀 สแกนและดึงคะแนน:   0%|          | 0/300 [00:00<?, ?it/s]


✅ เสร็จสมบูรณ์! ไฟล์ถูกเซฟไว้ที่: /content/drive/MyDrive/final_data/submission_easyocr_refactored.csv
📊 สรุป: ดึงคะแนนสำเร็จ 7638 แถว (คิดเป็น 75.98%) จากทั้งหมด 10053 แถว
